# CIC-IDS-2017 multiclass model experiments

This notebook compares five real model families plus a Dummy sanity check using MLflow. Macro F1 is the only primary model-selection metric. It first establishes 71-feature baselines and then validates seven proposed feature removals. The fixed test set is created for later work but is never passed to a model in this notebook.

The workflow uses all 15 detailed labels. Results for extremely rare classes must be interpreted using their support; one correct or incorrect prediction is not a reliable performance estimate when support is only one or a few flows.

## 1. Imports, paths and experiment settings

The local tracking database is stored at `ml/mlflow.db`, and run artifacts are stored under `ml/mlruns/`. Both locations are ignored by Git. Random state 42 is used for every split and stochastic estimator.

In [1]:
import gc
import hashlib
import json
import platform
import tempfile
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight


def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "ml").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
PROCESSED_DATA_PATH = PROJECT_ROOT / "ml" / "data" / "processed" / "cicids2017_cleaned.parquet"
MLFLOW_DATABASE_PATH = PROJECT_ROOT / "ml" / "mlflow.db"
MLFLOW_ARTIFACT_ROOT = PROJECT_ROOT / "ml" / "mlruns"

EXPERIMENT_NAME = "cicids2017-multiclass-baselines"
RANDOM_STATE = 42
TEST_SIZE = 0.20
VALIDATION_SIZE_WITHIN_TRAINING = 0.20

LATENCY_WARMUP_RUNS = 50
LATENCY_MEASUREMENTS = 1_000
THROUGHPUT_BATCH_SIZE = 10_000
THROUGHPUT_REPETITIONS = 5

EXPECTED_ROWS = 2_824_752
EXPECTED_COLUMNS = 76
LABEL_ORDER = [
    "BENIGN",
    "Bot",
    "DDoS",
    "DoS GoldenEye",
    "DoS Hulk",
    "DoS Slowhttptest",
    "DoS slowloris",
    "FTP-Patator",
    "Heartbleed",
    "Infiltration",
    "PortScan",
    "SSH-Patator",
    "Web Attack - Brute Force",
    "Web Attack - Sql Injection",
    "Web Attack - XSS",
]

if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(
        "The cleaned parquet dataset is missing. Run 01_data_exploration.ipynb first."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed dataset: {PROCESSED_DATA_PATH}")
print(f"Random state: {RANDOM_STATE}")
print(f"Final dataset proportions: 64% fit, 16% validation, 20% test")

Project root: C:\Users\ademz\Desktop\9raya\DoS-Intrusion-Detection-System
Processed dataset: C:\Users\ademz\Desktop\9raya\DoS-Intrusion-Detection-System\ml\data\processed\cicids2017_cleaned.parquet
Random state: 42
Final dataset proportions: 64% fit, 16% validation, 20% test


### Configure MLflow and identify the dataset

The SHA-256 digest identifies the exact cleaned parquet file used by every run. MLflow uses SQLite for run metadata and a local artifact directory for reports and figures.

In [2]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        while chunk := file_handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


MLFLOW_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
TRACKING_URI = f"sqlite:///{MLFLOW_DATABASE_PATH.as_posix()}"
mlflow.set_tracking_uri(TRACKING_URI)

existing_experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if existing_experiment is None:
    mlflow.create_experiment(
        EXPERIMENT_NAME,
        artifact_location=MLFLOW_ARTIFACT_ROOT.resolve().as_uri(),
    )
mlflow.set_experiment(EXPERIMENT_NAME)

DATASET_SHA256 = sha256_file(PROCESSED_DATA_PATH)
DATASET_VERSION = f"sha256:{DATASET_SHA256}"
environment_summary = pd.Series(
    {
        "Python": platform.python_version(),
        "scikit-learn": sklearn.__version__,
        "MLflow": mlflow.__version__,
        "Operating system": platform.platform(),
        "Dataset SHA-256": DATASET_SHA256,
        "Tracking URI": mlflow.get_tracking_uri(),
        "Artifact root": str(MLFLOW_ARTIFACT_ROOT),
    },
    name="Value",
)
display(environment_summary.to_frame())
print("To inspect runs after executing the notebook, start the UI with:")
print(f'mlflow ui --backend-store-uri "{TRACKING_URI}"')

2026/08/08 18:49:48 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/08/08 18:49:48 INFO mlflow.store.db.utils: Updating database tables


,Value
Python,3.10.11
scikit-learn,1.7.2
MLflow,3.15.1
Operating system,Windows-10-10.0.26200-SP0
Dataset SHA-256,28b946c70e39b7747302f1fac29f0ff73a20cdf83fd39d...
Tracking URI,sqlite:///C:/Users/ademz/Desktop/9raya/DoS-Int...
Artifact root,C:\Users\ademz\Desktop\9raya\DoS-Intrusion-Det...


To inspect runs after executing the notebook, start the UI with:
mlflow ui --backend-store-uri "sqlite:///C:/Users/ademz/Desktop/9raya/DoS-Intrusion-Detection-System/ml/mlflow.db"


## 2. Load and validate the cleaned dataset

The assertions prevent experiments from silently running on a different cleaning result or label set.

In [3]:
data = pd.read_parquet(PROCESSED_DATA_PATH)

dataset_checks = pd.Series(
    {
        "Rows": data.shape[0],
        "Columns": data.shape[1],
        "Target classes": data["Label"].nunique(dropna=False) if "Label" in data else 0,
        "Missing target labels": data["Label"].isna().sum() if "Label" in data else len(data),
        "Missing feature values": data.drop(columns=["Label"], errors="ignore").isna().sum().sum(),
    },
    name="Observed",
)
display(dataset_checks.to_frame())

if data.shape != (EXPECTED_ROWS, EXPECTED_COLUMNS):
    raise AssertionError(
        f"Expected {(EXPECTED_ROWS, EXPECTED_COLUMNS)}, observed {data.shape}."
    )
if data["Label"].isna().any():
    raise AssertionError("The target contains missing labels.")
if set(data["Label"].unique()) != set(LABEL_ORDER):
    raise AssertionError("The detailed label set differs from the expected 15 classes.")
if data.drop(columns="Label").isna().any().any():
    raise AssertionError("The cleaned feature data contains missing values.")

print("The cleaned dataset matches the expected EDA output.")

,Observed
Rows,2824752
Columns,76
Target classes,15
Missing target labels,0
Missing feature values,0


The cleaned dataset matches the expected EDA output.


## 3. Define the eligible feature schemas

Identifiers and the timestamp are excluded before splitting because they can encourage laboratory-specific memorization or reveal the published attack schedule. The first experiment uses all 71 eligible source features. A later paired experiment tests the same models after the seven documented redundancy candidates are removed.

In [4]:
IDENTIFIER_COLUMNS = [
    "Flow ID",
    "Src IP",
    "Dst IP",
    "Timestamp",
]
REDUNDANT_FEATURES = [
    "Fwd Segment Size Avg",
    "Bwd Segment Size Avg",
    "Subflow Fwd Packets",
    "Subflow Bwd Packets",
    "Subflow Fwd Bytes",
    "Subflow Bwd Bytes",
    "Packet Length Variance",
]

missing_identifiers = [column for column in IDENTIFIER_COLUMNS if column not in data]
missing_redundant_features = [column for column in REDUNDANT_FEATURES if column not in data]
if missing_identifiers or missing_redundant_features:
    raise KeyError(
        f"Missing identifiers: {missing_identifiers}; "
        f"missing redundancy decisions: {missing_redundant_features}"
    )

y = data.pop("Label")
X = data.drop(columns=IDENTIFIER_COLUMNS)
del data

MODEL_INPUT_FEATURES = X.columns.tolist()
REDUCED_FEATURES = [
    column for column in MODEL_INPUT_FEATURES if column not in REDUNDANT_FEATURES
]
FEATURE_SETS = {
    "all_71": MODEL_INPUT_FEATURES,
    "reduced_64": REDUCED_FEATURES,
}
PROTOCOL_VALUES = sorted(X["Protocol"].unique().tolist())
EXPECTED_PROTOCOL_VALUES = [0, 6, 17]
EXPECTED_TRANSFORMED_FEATURE_COUNTS = {
    feature_set: len(features) - 1 + len(EXPECTED_PROTOCOL_VALUES)
    for feature_set, features in FEATURE_SETS.items()
}

if len(MODEL_INPUT_FEATURES) != 71:
    raise AssertionError(f"Expected 71 model-input features, found {len(MODEL_INPUT_FEATURES)}.")
if len(REDUCED_FEATURES) != 64:
    raise AssertionError(f"Expected 64 reduced features, found {len(REDUCED_FEATURES)}.")
if PROTOCOL_VALUES != EXPECTED_PROTOCOL_VALUES:
    raise AssertionError(
        f"Expected Protocol values {EXPECTED_PROTOCOL_VALUES}, observed {PROTOCOL_VALUES}."
    )
if EXPECTED_TRANSFORMED_FEATURE_COUNTS != {"all_71": 73, "reduced_64": 66}:
    raise AssertionError("Unexpected transformed feature counts.")
if X.columns.duplicated().any():
    raise AssertionError("The model-input schema contains duplicate column names.")

feature_set_summary = pd.DataFrame(
    {
        "Source features": {
            name: len(features) for name, features in FEATURE_SETS.items()
        },
        "Transformed features after Protocol encoding": (
            EXPECTED_TRANSFORMED_FEATURE_COUNTS
        ),
    }
)
display(feature_set_summary)
print(f"Protocol values encoded inside every pipeline: {PROTOCOL_VALUES}")

,Source features,Transformed features after Protocol encoding
all_71,71,73
reduced_64,64,66


Protocol values encoded inside every pipeline: [0.0, 6.0, 17.0]


`Protocol` remains numeric in `X`, but every pipeline one-hot encodes it because the values identify protocol names rather than ordered quantities. The other features pass through unchanged for Dummy and tree models; SGD and MLP scale them using statistics fitted only on the fitting partition.

## 4. Create fixed fit, validation and test partitions

The first split exactly reproduces Notebook 2's stratified 80/20 training/test split. The 80% training partition is then divided into 80% fitting and 20% validation data, producing final proportions of 64%, 16% and 20%. Only the fit and validation partitions are used to select configurations.

In [5]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)
X_fit, X_validation, y_fit, y_validation = train_test_split(
    X_train_full,
    y_train_full,
    test_size=VALIDATION_SIZE_WITHIN_TRAINING,
    random_state=RANDOM_STATE,
    stratify=y_train_full,
)
del X, y

partition_indices = {
    "fit": X_fit.index,
    "validation": X_validation.index,
    "test": X_test.index,
}
for first_name, first_index in partition_indices.items():
    for second_name, second_index in partition_indices.items():
        if first_name < second_name and not first_index.intersection(second_index).empty:
            raise AssertionError(f"{first_name} and {second_name} rows overlap.")

def index_fingerprint(index):
    values = np.asarray(index, dtype=np.int64)
    return hashlib.sha256(values.tobytes()).hexdigest()


FIT_SPLIT_FINGERPRINT = index_fingerprint(X_fit.index)
VALIDATION_SPLIT_FINGERPRINT = index_fingerprint(X_validation.index)
TEST_SPLIT_FINGERPRINT = index_fingerprint(X_test.index)

partition_targets = {
    "Fit": y_fit,
    "Validation": y_validation,
    "Test": y_test,
}
for partition_name, target in partition_targets.items():
    missing_labels = set(LABEL_ORDER) - set(target.unique())
    if missing_labels:
        raise AssertionError(f"{partition_name} is missing labels: {sorted(missing_labels)}")

split_distribution = pd.DataFrame(
    {name: target.value_counts().reindex(LABEL_ORDER, fill_value=0)
     for name, target in partition_targets.items()}
)
split_distribution["Total"] = split_distribution.sum(axis=1)
display(split_distribution)

rare_class_support = split_distribution.loc[split_distribution["Total"] < 100]
print("Classes with fewer than 100 total flows:")
display(rare_class_support)
print("All 15 labels are present in every partition, and no row indices overlap.")

,Fit,Validation,Test,Total
Label,,,,
BENIGN,1451769,362943,453679,2268391
Bot,1252,313,391,1956
DDoS,81924,20481,25601,128006
DoS GoldenEye,6584,1646,2058,10288
DoS Hulk,147177,36794,45993,229964
DoS Slowhttptest,3519,880,1100,5499
DoS slowloris,3709,928,1159,5796
FTP-Patator,5076,1269,1586,7931
Heartbleed,5,1,1,7


Classes with fewer than 100 total flows:


,Fit,Validation,Test,Total
Label,,,,
Heartbleed,5,1,1,7
Infiltration,22,6,7,35
Web Attack - Sql Injection,14,3,4,21


All 15 labels are present in every partition, and no row indices overlap.


The rare-class table is a mandatory warning, not a reason to merge labels. In particular, Heartbleed has too little support for its validation or eventual test score to be treated as a stable estimate. The test features and labels are not passed to any model in this notebook.

## 5. Define preprocessing pipelines and model configurations

The Dummy model is a sanity check. SGD is the low-latency linear baseline. The Decision Tree is a regularized nonlinear baseline. Random Forest tests bagging, Histogram Gradient Boosting tests sequential boosted trees, and a small MLP represents neural networks for tabular flows. Every real model has an unweighted and balanced configuration; SMOTE is not used.

In [6]:
BASELINE_FEATURE_SET = "all_71"
MODEL_SPECS = [
    {"model_key": "dummy", "model_family": "DummyClassifier", "weighting_mode": "unweighted", "feature_set": BASELINE_FEATURE_SET},
    {"model_key": "sgd", "model_family": "SGDClassifier", "weighting_mode": "unweighted", "feature_set": BASELINE_FEATURE_SET},
    {"model_key": "sgd", "model_family": "SGDClassifier", "weighting_mode": "balanced", "feature_set": BASELINE_FEATURE_SET},
    {"model_key": "decision_tree", "model_family": "DecisionTreeClassifier", "weighting_mode": "unweighted", "feature_set": BASELINE_FEATURE_SET},
    {"model_key": "decision_tree", "model_family": "DecisionTreeClassifier", "weighting_mode": "balanced", "feature_set": BASELINE_FEATURE_SET},
    {"model_key": "random_forest", "model_family": "RandomForestClassifier", "weighting_mode": "unweighted", "feature_set": BASELINE_FEATURE_SET},
    {"model_key": "random_forest", "model_family": "RandomForestClassifier", "weighting_mode": "balanced", "feature_set": BASELINE_FEATURE_SET},
    {"model_key": "hist_gradient_boosting", "model_family": "HistGradientBoostingClassifier", "weighting_mode": "unweighted", "feature_set": BASELINE_FEATURE_SET},
    {"model_key": "hist_gradient_boosting", "model_family": "HistGradientBoostingClassifier", "weighting_mode": "balanced", "feature_set": BASELINE_FEATURE_SET},
    {"model_key": "mlp", "model_family": "MLPClassifier", "weighting_mode": "unweighted", "feature_set": BASELINE_FEATURE_SET},
    {"model_key": "mlp", "model_family": "MLPClassifier", "weighting_mode": "balanced", "feature_set": BASELINE_FEATURE_SET},
]

def make_preprocessor(feature_set, scale_numeric):
    if feature_set not in FEATURE_SETS:
        raise ValueError(f"Unknown feature set: {feature_set}")
    selected_features = FEATURE_SETS[feature_set]
    numeric_features = [
        feature for feature in selected_features if feature != "Protocol"
    ]
    numeric_transformer = StandardScaler() if scale_numeric else "passthrough"
    return ColumnTransformer(
        [
            ("numeric", numeric_transformer, numeric_features),
            (
                "protocol",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                ["Protocol"],
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


def build_pipeline(spec):
    weighted = spec["weighting_mode"] == "balanced"
    model_key = spec["model_key"]
    scale_numeric = model_key in {"sgd", "mlp"}
    preprocessor = make_preprocessor(spec["feature_set"], scale_numeric)

    if model_key == "dummy":
        classifier = DummyClassifier(strategy="most_frequent")
    elif model_key == "sgd":
        classifier = SGDClassifier(
            loss="log_loss",
            class_weight="balanced" if weighted else None,
            max_iter=1_000,
            tol=1e-3,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    elif model_key == "decision_tree":
        classifier = DecisionTreeClassifier(
            max_depth=20,
            min_samples_leaf=5,
            class_weight="balanced" if weighted else None,
            random_state=RANDOM_STATE,
        )
    elif model_key == "random_forest":
        classifier = RandomForestClassifier(
            n_estimators=100,
            max_depth=20,
            min_samples_leaf=2,
            max_features="sqrt",
            class_weight="balanced_subsample" if weighted else None,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    elif model_key == "hist_gradient_boosting":
        classifier = HistGradientBoostingClassifier(
            learning_rate=0.1,
            max_iter=100,
            max_leaf_nodes=31,
            l2_regularization=1.0,
            class_weight="balanced" if weighted else None,
            early_stopping=True,
            random_state=RANDOM_STATE,
        )
    elif model_key == "mlp":
        classifier = MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            batch_size=2_048,
            learning_rate_init=1e-3,
            max_iter=50,
            early_stopping=True,
            validation_fraction=0.10,
            n_iter_no_change=5,
            random_state=RANDOM_STATE,
        )
    else:
        raise ValueError(f"Unknown model key: {model_key}")

    return Pipeline(
        [
            ("preprocessor", preprocessor),
            ("classifier", classifier),
        ]
    )


model_plan = pd.DataFrame(MODEL_SPECS)
model_plan.index = pd.RangeIndex(1, len(model_plan) + 1, name="Run order")
display(model_plan)

,model_key,model_family,weighting_mode,feature_set
Run order,,,,
1,dummy,DummyClassifier,unweighted,all_71
2,sgd,SGDClassifier,unweighted,all_71
3,sgd,SGDClassifier,balanced,all_71
4,decision_tree,DecisionTreeClassifier,unweighted,all_71
5,decision_tree,DecisionTreeClassifier,balanced,all_71
6,random_forest,RandomForestClassifier,unweighted,all_71
7,random_forest,RandomForestClassifier,balanced,all_71
8,hist_gradient_boosting,HistGradientBoostingClassifier,unweighted,all_71
9,hist_gradient_boosting,HistGradientBoostingClassifier,balanced,all_71


The initial parameters are controlled baselines, not a hyperparameter search. Comparing weighting modes isolates whether inverse-frequency weighting helps rare classes enough to offset any loss on common classes. The balanced MLP receives sample weights calculated only from the fitting labels. For compatibility with MLP early stopping, its string targets are mapped internally to the fixed `LABEL_ORDER` indices and its predictions are decoded before evaluation; the dataset labels are not changed. XGBoost is deliberately postponed until this complete evaluation path has been validated.

## 6. Define evaluation, artifact and timing utilities

Every call uses the label order from Section 1. Macro F1 is calculated normally and independently reconstructed from the confusion matrix. A mismatch raises an error. Latency times only the already-created input flowing through the complete fitted pipeline; dataframe sampling is performed before timing begins.

In [7]:
def calculate_metrics_and_diagnostics(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        zero_division=0,
    )
    report = pd.DataFrame(
        {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support.astype(np.int64),
        },
        index=pd.Index(LABEL_ORDER, name="label"),
    )

    raw_matrix = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    row_totals = raw_matrix.sum(axis=1, keepdims=True)
    normalized_matrix = np.divide(
        raw_matrix,
        row_totals,
        out=np.zeros_like(raw_matrix, dtype=float),
        where=row_totals != 0,
    )
    nonempty_rows = row_totals.ravel() > 0
    if not np.allclose(normalized_matrix[nonempty_rows].sum(axis=1), 1.0):
        raise AssertionError("A non-empty normalized confusion-matrix row does not sum to 1.")

    true_positive = np.diag(raw_matrix).astype(float)
    false_positive = raw_matrix.sum(axis=0) - true_positive
    false_negative = raw_matrix.sum(axis=1) - true_positive
    f1_denominator = 2 * true_positive + false_positive + false_negative
    f1_from_matrix = np.divide(
        2 * true_positive,
        f1_denominator,
        out=np.zeros_like(true_positive),
        where=f1_denominator != 0,
    )
    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        average="macro",
        zero_division=0,
    )
    independently_recalculated_macro_f1 = f1_from_matrix.mean()
    if not np.isclose(macro_f1, independently_recalculated_macro_f1):
        raise AssertionError("Macro F1 does not match the confusion-matrix calculation.")

    y_true_attack = np.asarray(y_true) != "BENIGN"
    y_pred_attack = np.asarray(y_pred) != "BENIGN"
    scalar_metrics = {
        "macro_f1": float(macro_f1),
        "accuracy_reference": float(accuracy_score(y_true, y_pred)),
        "binary_attack_recall": float(
            recall_score(y_true_attack, y_pred_attack, zero_division=0)
        ),
    }
    return scalar_metrics, report, raw_matrix, normalized_matrix


def make_timing_inputs(X_source):
    generator = np.random.default_rng(RANDOM_STATE)
    latency_count = min(LATENCY_MEASUREMENTS, len(X_source))
    latency_positions = generator.choice(len(X_source), size=latency_count, replace=False)
    single_rows = tuple(X_source.iloc[[int(position)]].copy() for position in latency_positions)

    batch_count = min(THROUGHPUT_BATCH_SIZE, len(X_source))
    batch_positions = generator.choice(len(X_source), size=batch_count, replace=False)
    batch_rows = X_source.iloc[batch_positions].copy()
    return single_rows, batch_rows


def timing_input_fingerprint(timing_inputs):
    single_rows, batch_rows = timing_inputs
    latency_indices = np.asarray(
        [int(row.index[0]) for row in single_rows],
        dtype=np.int64,
    )
    batch_indices = np.asarray(batch_rows.index, dtype=np.int64)
    digest = hashlib.sha256()
    digest.update(latency_indices.tobytes())
    digest.update(batch_indices.tobytes())
    return digest.hexdigest()


def measure_pipeline_speed(pipeline, single_rows, batch_rows):
    warmup_row = single_rows[0]
    for _ in range(LATENCY_WARMUP_RUNS):
        pipeline.predict(warmup_row)

    latency_ms = []
    for row in single_rows:
        start = perf_counter()
        pipeline.predict(row)
        latency_ms.append((perf_counter() - start) * 1_000)

    pipeline.predict(batch_rows)
    batch_start = perf_counter()
    for _ in range(THROUGHPUT_REPETITIONS):
        pipeline.predict(batch_rows)
    batch_seconds = perf_counter() - batch_start

    return {
        "latency_p50_ms": float(np.percentile(latency_ms, 50)),
        "latency_p95_ms": float(np.percentile(latency_ms, 95)),
        "latency_p99_ms": float(np.percentile(latency_ms, 99)),
        "throughput_flows_per_second": float(
            len(batch_rows) * THROUGHPUT_REPETITIONS / batch_seconds
        ),
    }


def save_confusion_figure(matrix, output_path, title, value_format):
    figure, axis = plt.subplots(figsize=(15, 12))
    sns.heatmap(
        matrix,
        annot=True,
        fmt=value_format,
        cmap="Blues",
        xticklabels=LABEL_ORDER,
        yticklabels=LABEL_ORDER,
        ax=axis,
    )
    axis.set_title(title)
    axis.set_xlabel("Predicted label")
    axis.set_ylabel("True label")
    axis.tick_params(axis="x", labelrotation=45)
    axis.tick_params(axis="y", labelrotation=0)
    figure.tight_layout()
    figure.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(figure)


def log_evaluation_artifacts(report, raw_matrix, normalized_matrix, run_context):
    raw_frame = pd.DataFrame(raw_matrix, index=LABEL_ORDER, columns=LABEL_ORDER)
    normalized_frame = pd.DataFrame(
        normalized_matrix, index=LABEL_ORDER, columns=LABEL_ORDER
    )

    with tempfile.TemporaryDirectory() as temporary_directory:
        report_directory = Path(temporary_directory)
        report.to_csv(report_directory / "per_class_report.csv")
        raw_frame.to_csv(report_directory / "confusion_matrix_raw.csv")
        normalized_frame.to_csv(
            report_directory / "confusion_matrix_row_normalized.csv"
        )
        save_confusion_figure(
            raw_matrix,
            report_directory / "confusion_matrix_raw.png",
            "Raw multiclass confusion matrix",
            "d",
        )
        save_confusion_figure(
            normalized_matrix,
            report_directory / "confusion_matrix_row_normalized.png",
            "Row-normalized multiclass confusion matrix",
            ".3f",
        )
        with (report_directory / "run_context.json").open("w", encoding="utf-8") as file_handle:
            json.dump(run_context, file_handle, indent=2)
        mlflow.log_artifacts(report_directory, artifact_path="evaluation")


def log_pipeline_parameters(
    pipeline, spec, weighting_mechanism, target_encoding
):
    source_feature_count = len(FEATURE_SETS[spec["feature_set"]])
    parameters = {
        "model_key": spec["model_key"],
        "weighting_mode": spec["weighting_mode"],
        "weighting_mechanism": weighting_mechanism,
        "target_encoding": target_encoding,
        "feature_set": spec["feature_set"],
        "source_feature_count": source_feature_count,
        "expected_transformed_feature_count": (
            EXPECTED_TRANSFORMED_FEATURE_COUNTS[spec["feature_set"]]
        ),
        "random_state": RANDOM_STATE,
    }
    for name, value in pipeline.named_steps["classifier"].get_params().items():
        if isinstance(value, (str, int, float, bool)) or value is None:
            parameters[f"classifier__{name}"] = value
    mlflow.log_params(parameters)


def make_fit_parameters(spec, y_training):
    if spec["model_key"] == "mlp" and spec["weighting_mode"] == "balanced":
        sample_weights = compute_sample_weight(
            class_weight="balanced",
            y=y_training,
        )
        if len(sample_weights) != len(y_training):
            raise AssertionError("MLP sample weights do not match the fitting labels.")
        return {"classifier__sample_weight": sample_weights}, "balanced_sample_weight"
    if spec["weighting_mode"] == "balanced":
        return {}, "classifier_class_weight"
    return {}, "none"


def run_experiment(
    spec,
    X_training,
    y_training,
    X_evaluation,
    y_evaluation,
    timing_inputs,
    evaluation_stage,
    fit_split_fingerprint,
    evaluation_split_fingerprint,
):
    pipeline = build_pipeline(spec)
    source_features = FEATURE_SETS[spec["feature_set"]]
    source_feature_count = len(source_features)
    expected_transformed_feature_count = (
        EXPECTED_TRANSFORMED_FEATURE_COUNTS[spec["feature_set"]]
    )
    timing_fingerprint = timing_input_fingerprint(timing_inputs)
    fit_parameters, weighting_mechanism = make_fit_parameters(spec, y_training)
    if spec["model_key"] == "mlp":
        label_to_index = {label: index for index, label in enumerate(LABEL_ORDER)}
        fitting_target = y_training.map(label_to_index)
        if fitting_target.isna().any():
            raise AssertionError("MLP target encoding encountered an unknown label.")
        fitting_target = fitting_target.astype(np.int64)
        target_encoding = "fixed_label_index_for_mlp"
    else:
        fitting_target = y_training
        target_encoding = "original_string_labels"
    run_name = (
        f"{evaluation_stage}__{spec['model_key']}__"
        f"{spec['weighting_mode']}__{spec['feature_set']}"
    )
    tags = {
        "model_family": spec["model_family"],
        "weighting_mode": spec["weighting_mode"],
        "weighting_mechanism": weighting_mechanism,
        "target_encoding": target_encoding,
        "split_seed": str(RANDOM_STATE),
        "feature_set": spec["feature_set"],
        "source_feature_count": str(source_feature_count),
        "expected_transformed_feature_count": str(
            expected_transformed_feature_count
        ),
        "dataset_version": DATASET_VERSION,
        "evaluation_stage": evaluation_stage,
        "fit_split_fingerprint": fit_split_fingerprint,
        "evaluation_split_fingerprint": evaluation_split_fingerprint,
        "timing_input_fingerprint": timing_fingerprint,
    }

    with mlflow.start_run(run_name=run_name, tags=tags) as active_run:
        log_pipeline_parameters(
            pipeline, spec, weighting_mechanism, target_encoding
        )

        training_start = perf_counter()
        pipeline.fit(X_training, fitting_target, **fit_parameters)
        training_time_seconds = perf_counter() - training_start
        actual_transformed_feature_count = len(
            pipeline.named_steps["preprocessor"].get_feature_names_out()
        )
        if actual_transformed_feature_count != expected_transformed_feature_count:
            raise AssertionError(
                "Expected "
                f"{expected_transformed_feature_count} transformed features, "
                f"observed {actual_transformed_feature_count}."
            )
        learned_protocol_values = (
            pipeline.named_steps["preprocessor"]
            .named_transformers_["protocol"]
            .categories_[0]
            .tolist()
        )
        if learned_protocol_values != EXPECTED_PROTOCOL_VALUES:
            raise AssertionError(
                f"Unexpected fitted Protocol categories: {learned_protocol_values}"
            )

        predictions = pipeline.predict(X_evaluation)
        if spec["model_key"] == "mlp":
            prediction_indices = np.asarray(predictions, dtype=np.int64)
            if ((prediction_indices < 0) | (prediction_indices >= len(LABEL_ORDER))).any():
                raise AssertionError("MLP predicted an unknown encoded label.")
            predictions = np.asarray(LABEL_ORDER, dtype=object)[prediction_indices]
        metrics, report, raw_matrix, normalized_matrix = (
            calculate_metrics_and_diagnostics(y_evaluation, predictions)
        )
        metrics["training_time_seconds"] = float(training_time_seconds)
        metrics.update(measure_pipeline_speed(pipeline, *timing_inputs))

        run_context = {
            "dataset_sha256": DATASET_SHA256,
            "label_order": LABEL_ORDER,
            "pipeline_input_features": MODEL_INPUT_FEATURES,
            "feature_set": spec["feature_set"],
            "selected_source_features": source_features,
            "source_feature_count": source_feature_count,
            "transformed_feature_count": actual_transformed_feature_count,
            "one_hot_protocol_values": learned_protocol_values,
            "target_encoding": target_encoding,
            "proposed_redundant_features": REDUNDANT_FEATURES,
            "evaluation_stage": evaluation_stage,
            "fit_rows": len(X_training),
            "evaluation_rows": len(X_evaluation),
            "latency_warmup_runs": LATENCY_WARMUP_RUNS,
            "latency_measurements": len(timing_inputs[0]),
            "throughput_batch_size": len(timing_inputs[1]),
            "throughput_repetitions": THROUGHPUT_REPETITIONS,
            "timing_input_fingerprint": timing_fingerprint,
        }

        mlflow.log_metrics(metrics)
        log_evaluation_artifacts(
            report, raw_matrix, normalized_matrix, run_context
        )

        result = {
            "run_id": active_run.info.run_id,
            "model_key": spec["model_key"],
            "model_family": spec["model_family"],
            "weighting_mode": spec["weighting_mode"],
            "weighting_mechanism": weighting_mechanism,
            "target_encoding": target_encoding,
            "feature_set": spec["feature_set"],
            "source_feature_count": source_feature_count,
            "transformed_feature_count": actual_transformed_feature_count,
            "evaluation_stage": evaluation_stage,
            "fit_split_fingerprint": fit_split_fingerprint,
            "evaluation_split_fingerprint": evaluation_split_fingerprint,
            "timing_input_fingerprint": timing_fingerprint,
            **metrics,
        }

    del pipeline, predictions
    gc.collect()
    return result

The per-class artifact contains only precision, recall, F1 and support. Both confusion matrices are logged as CSV and PNG files. No model is serialized here because feature, model and tuning decisions are not final.

## 7. Run and rank the validation experiments

This section fits all 11 planned 71-feature configurations on the 64% fit partition and evaluates them on the fixed 16% validation partition. Depending on the machine, the forest, boosting and MLP runs can take substantial time. The same preselected timing inputs are used for every configuration.

In [8]:
validation_timing_inputs = make_timing_inputs(X_validation)
validation_results_records = []

for run_number, spec in enumerate(MODEL_SPECS, start=1):
    print(
        f"[{run_number}/{len(MODEL_SPECS)}] "
        f"{spec['model_family']} ({spec['weighting_mode']})"
    )
    result = run_experiment(
        spec=spec,
        X_training=X_fit,
        y_training=y_fit,
        X_evaluation=X_validation,
        y_evaluation=y_validation,
        timing_inputs=validation_timing_inputs,
        evaluation_stage="validation",
        fit_split_fingerprint=FIT_SPLIT_FINGERPRINT,
        evaluation_split_fingerprint=VALIDATION_SPLIT_FINGERPRINT,
    )
    validation_results_records.append(result)
    print(f"Validation macro F1: {result['macro_f1']:.6f}")

validation_results = (
    pd.DataFrame(validation_results_records)
    .sort_values("macro_f1", ascending=False)
    .reset_index(drop=True)
)
validation_results.index = pd.RangeIndex(1, len(validation_results) + 1, name="Validation rank")
display(validation_results)

[1/11] DummyClassifier (unweighted)


Validation macro F1: 0.059384
[2/11] SGDClassifier (unweighted)


Validation macro F1: 0.391504
[3/11] SGDClassifier (balanced)


Validation macro F1: 0.452873
[4/11] DecisionTreeClassifier (unweighted)


Validation macro F1: 0.809482
[5/11] DecisionTreeClassifier (balanced)


...

Validation macro F1: 0.950572
[9/11] HistGradientBoostingClassifier (balanced)


Validation macro F1: 0.961751
[10/11] MLPClassifier (unweighted)


Validation macro F1: 0.696669
[11/11] MLPClassifier (balanced)


Validation macro F1: 0.597323


,run_id,model_key,model_family,weighting_mode,weighting_mechanism,target_encoding,feature_set,source_feature_count,transformed_feature_count,evaluation_stage,...,evaluation_split_fingerprint,timing_input_fingerprint,macro_f1,accuracy_reference,binary_attack_recall,training_time_seconds,latency_p50_ms,latency_p95_ms,latency_p99_ms,throughput_flows_per_second
Validation rank,,,,,,,,,,,,,,,,,,,,,
1,672b352e49c04ad7bedead13035f695c,hist_gradient_boosting,HistGradientBoostingClassifier,balanced,classifier_class_weight,original_string_labels,all_71,71,73,validation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.961751,0.999296,0.999865,86.104058,11.72800,17.643455,18.953830,1.038161e+05
2,5a8654a12b534ae4b1c0496816f4ed78,hist_gradient_boosting,HistGradientBoostingClassifier,unweighted,none,original_string_labels,all_71,71,73,validation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.950572,0.999502,0.999315,157.646561,18.70190,27.715310,28.950075,6.053683e+04
3,556c47f26e9740abb748777aee46e419,decision_tree,DecisionTreeClassifier,balanced,classifier_class_weight,original_string_labels,all_71,71,73,validation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.942137,0.998951,0.999539,82.180333,1.17575,1.555635,1.851507,1.288789e+06
4,b5c29cbe221d4a20ad8f08cb29b51a25,random_forest,RandomForestClassifier,balanced,classifier_class_weight,original_string_labels,all_71,71,73,validation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.895785,0.998821,0.999551,171.334838,16.68040,20.201050,33.956476,2.000011e+05
5,6113b02899214082abe8468144d2d56d,random_forest,RandomForestClassifier,unweighted,none,original_string_labels,all_71,71,73,validation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.856176,0.999279,0.998360,180.747684,16.68835,17.945610,33.786842,1.996887e+05
6,4a8bd2b338524bc696eaa8d2091b3a3b,decision_tree,DecisionTreeClassifier,unweighted,none,original_string_labels,all_71,71,73,validation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.809482,0.998774,0.995664,96.310912,1.18565,1.471765,1.708117,1.467317e+06
7,cfb05a1d134e481aa5e4e1ee059d38ac,mlp,MLPClassifier,unweighted,none,fixed_label_index_for_mlp,all_71,71,73,validation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.696669,0.993723,0.989317,106.032988,1.65590,1.979505,2.276155,5.555241e+05
8,f27d5257554647bebe366c179cacd308,mlp,MLPClassifier,balanced,balanced_sample_weight,fixed_label_index_for_mlp,all_71,71,73,validation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.597323,0.955372,0.998528,176.745318,1.67370,2.100475,2.362549,5.708564e+05
9,92d4adc9a14a45b0b7151b35f487024d,sgd,SGDClassifier,balanced,classifier_class_weight,original_string_labels,all_71,71,73,validation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.452873,0.918829,0.985980,56.255509,1.60425,2.026405,2.399724,1.072248e+06


### Validate the complete 71-feature baseline matrix

Every real configuration must beat the Dummy macro F1. All 11 baseline runs must also show the same fit, validation and timing fingerprints. Section 8 will mirror the ten real-model configurations with the reduced feature set; Dummy remains a single fixed reference.

In [9]:
dummy_rows = validation_results.loc[
    validation_results["model_family"] == "DummyClassifier"
]
if len(dummy_rows) != 1:
    raise AssertionError("Expected exactly one Dummy validation run.")
dummy_macro_f1 = dummy_rows.iloc[0]["macro_f1"]
real_results = validation_results.loc[
    validation_results["model_family"] != "DummyClassifier"
]
if not real_results["macro_f1"].gt(dummy_macro_f1).all():
    failed = real_results.loc[
        ~real_results["macro_f1"].gt(dummy_macro_f1),
        ["model_family", "weighting_mode", "macro_f1"],
    ]
    raise AssertionError(f"Real configurations did not beat Dummy:\n{failed}")

if validation_results["fit_split_fingerprint"].nunique() != 1:
    raise AssertionError("Validation runs used different fit rows.")
if validation_results["evaluation_split_fingerprint"].nunique() != 1:
    raise AssertionError("Validation runs used different validation rows.")

if len(validation_results) != 11:
    raise AssertionError("Expected 11 baseline validation runs.")
if validation_results["feature_set"].ne("all_71").any():
    raise AssertionError("Every baseline must use all 71 source features.")
if validation_results["source_feature_count"].ne(71).any():
    raise AssertionError("A baseline run has the wrong source feature count.")
if validation_results["transformed_feature_count"].ne(73).any():
    raise AssertionError("A baseline run has the wrong transformed feature count.")

if validation_results[["model_key", "weighting_mode"]].duplicated().any():
    raise AssertionError("The 71-feature baseline matrix contains a duplicate configuration.")
print("All ten real baseline configurations are ready for paired 64-feature runs.")
print("Dummy remains a single 71-feature reference run.")
print("The test set has not been evaluated.")

All ten real baseline configurations are ready for paired 64-feature runs.
Dummy remains a single 71-feature reference run.
The test set has not been evaluated.


The baseline table is descriptive at this stage; it does not eliminate a weighting mode or model family. Review the per-class MLflow artifacts before interpreting why configurations differ.

## 8. Validate the seven proposed feature removals

All ten real-model configurations are rerun with the seven proposed redundancy candidates removed. The fit rows, validation rows and timing inputs remain identical to Section 7, producing a complete paired 71-versus-64 comparison that can reveal interactions between weighting and feature removal. Dummy is not repeated because its predictions ignore feature values.

In [10]:
ablation_specs = [
    {**spec, "feature_set": "reduced_64"}
    for spec in MODEL_SPECS
    if spec["model_key"] != "dummy"
]
if len(ablation_specs) != 10:
    raise AssertionError("Expected ten reduced-feature real-model configurations.")
feature_ablation_records = []

for run_number, spec in enumerate(ablation_specs, start=1):
    print(
        f"[{run_number}/{len(ablation_specs)}] "
        f"{spec['model_family']} ({spec['weighting_mode']}, reduced_64)"
    )
    result = run_experiment(
        spec=spec,
        X_training=X_fit,
        y_training=y_fit,
        X_evaluation=X_validation,
        y_evaluation=y_validation,
        timing_inputs=validation_timing_inputs,
        evaluation_stage="feature_ablation",
        fit_split_fingerprint=FIT_SPLIT_FINGERPRINT,
        evaluation_split_fingerprint=VALIDATION_SPLIT_FINGERPRINT,
    )
    feature_ablation_records.append(result)
    print(f"Reduced-feature validation macro F1: {result['macro_f1']:.6f}")

feature_ablation_results = pd.DataFrame(feature_ablation_records)
display(feature_ablation_results)

[1/10] SGDClassifier (unweighted, reduced_64)


Reduced-feature validation macro F1: 0.383278
[2/10] SGDClassifier (balanced, reduced_64)


Reduced-feature validation macro F1: 0.447660
[3/10] DecisionTreeClassifier (unweighted, reduced_64)


Reduced-feature validation macro F1: 0.811481
[4/10] DecisionTreeClassifier (balanced, reduced_64)


Reduced-feature validation macro F1: 0.942255
[5/10] RandomForestClassifier (unweighted, reduced_64)


...

Reduced-feature validation macro F1: 0.950144
[8/10] HistGradientBoostingClassifier (balanced, reduced_64)


Reduced-feature validation macro F1: 0.967029
[9/10] MLPClassifier (unweighted, reduced_64)


Reduced-feature validation macro F1: 0.779481
[10/10] MLPClassifier (balanced, reduced_64)


Reduced-feature validation macro F1: 0.625217


,run_id,model_key,model_family,weighting_mode,weighting_mechanism,target_encoding,feature_set,source_feature_count,transformed_feature_count,evaluation_stage,...,evaluation_split_fingerprint,timing_input_fingerprint,macro_f1,accuracy_reference,binary_attack_recall,training_time_seconds,latency_p50_ms,latency_p95_ms,latency_p99_ms,throughput_flows_per_second
0,5f4aff4cb50e46f6ae0ef16f6b51d708,sgd,SGDClassifier,unweighted,none,original_string_labels,reduced_64,64,66,feature_ablation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.383278,0.954144,0.881047,15.587546,1.64500,2.126720,2.636099,1.047430e+06
1,cddc80dee65645eeb67f65d7f3c89aec,sgd,SGDClassifier,balanced,classifier_class_weight,original_string_labels,reduced_64,64,66,feature_ablation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.447660,0.915249,0.985419,43.458967,1.58785,1.925045,2.213198,1.083334e+06
2,03c6738846414fe39dbf16a9bb2c5575,decision_tree,DecisionTreeClassifier,unweighted,none,original_string_labels,reduced_64,64,66,feature_ablation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.811481,0.998779,0.995641,87.947156,1.15660,1.458820,1.739735,1.490211e+06
3,bfd90d9524df44cf928d054a34a4b8a8,decision_tree,DecisionTreeClassifier,balanced,classifier_class_weight,original_string_labels,reduced_64,64,66,feature_ablation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.942255,0.998956,0.999528,70.078793,1.14950,1.302795,1.500515,1.608798e+06
4,4f00556d9b664614911921f2751add6e,random_forest,RandomForestClassifier,unweighted,none,original_string_labels,reduced_64,64,66,feature_ablation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.884431,0.999361,0.998573,168.517611,16.69850,31.957960,34.225024,1.999531e+05
5,e426da11cca14902a3e502cdd47c22f3,random_forest,RandomForestClassifier,balanced,classifier_class_weight,original_string_labels,reduced_64,64,66,feature_ablation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.948437,0.999022,0.999607,164.771264,16.71545,31.572785,33.906162,1.887238e+05
6,79abfa161e964811a7fbe3adf30849f5,hist_gradient_boosting,HistGradientBoostingClassifier,unweighted,none,original_string_labels,reduced_64,64,66,feature_ablation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.950144,0.999487,0.999281,141.672212,18.87760,28.204740,30.699401,5.931190e+04
7,53cbe1eeec8d40c597db63190993b6bd,hist_gradient_boosting,HistGradientBoostingClassifier,balanced,classifier_class_weight,original_string_labels,reduced_64,64,66,feature_ablation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.967029,0.999303,0.999854,83.829454,11.93740,18.309620,18.948630,1.003926e+05
8,b65293eea9d94ae6927dd685eae40c21,mlp,MLPClassifier,unweighted,none,fixed_label_index_for_mlp,reduced_64,64,66,feature_ablation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.779481,0.996759,0.995507,326.634468,1.63215,1.925755,2.256447,5.682270e+05
9,f6df06f2c8e944a1a31d3f6a472404df,mlp,MLPClassifier,balanced,balanced_sample_weight,fixed_label_index_for_mlp,reduced_64,64,66,feature_ablation,...,f21bc202dcdfb13964ca7cf323d7327212f2a5231e3715...,da802a73d33489107c9109db2a6b15637c8675ad597088...,0.625217,0.948473,0.999315,215.936173,1.66325,1.940435,2.138233,5.857119e+05


### Compare the paired feature sets

Each weighting configuration receives its own paired feature comparison. The strict pair-level rule prefers 64 source features only when its validation macro F1 equals or exceeds the corresponding 71-feature result. The final table then selects the highest-macro-F1 combination of feature set and weighting mode independently for each real model family. Operational differences never override the primary metric.

In [11]:
comparison_rows = []
for reduced_result in feature_ablation_results.itertuples(index=False):
    matching_baseline = validation_results.loc[
        (validation_results["model_family"] == reduced_result.model_family)
        & (validation_results["weighting_mode"] == reduced_result.weighting_mode)
    ]
    if len(matching_baseline) != 1:
        raise AssertionError("Expected exactly one paired 71-feature baseline.")
    baseline = matching_baseline.iloc[0]
    use_reduced_features = reduced_result.macro_f1 >= baseline["macro_f1"]
    comparison_rows.append(
        {
            "Model family": reduced_result.model_family,
            "Weighting mode": reduced_result.weighting_mode,
            "71-feature macro F1": baseline["macro_f1"],
            "64-feature macro F1": reduced_result.macro_f1,
            "Macro F1 difference (64 - 71)": (
                reduced_result.macro_f1 - baseline["macro_f1"]
            ),
            "71-feature p99 latency (ms)": baseline["latency_p99_ms"],
            "64-feature p99 latency (ms)": reduced_result.latency_p99_ms,
            "p99 latency difference (64 - 71)": (
                reduced_result.latency_p99_ms - baseline["latency_p99_ms"]
            ),
            "71-feature throughput": baseline["throughput_flows_per_second"],
            "64-feature throughput": reduced_result.throughput_flows_per_second,
            "Throughput difference (64 - 71)": (
                reduced_result.throughput_flows_per_second
                - baseline["throughput_flows_per_second"]
            ),
            "71 source features": int(baseline["source_feature_count"]),
            "71 transformed features": int(baseline["transformed_feature_count"]),
            "64 source features": int(reduced_result.source_feature_count),
            "64 transformed features": int(reduced_result.transformed_feature_count),
            "71-feature per-class artifact run ID": baseline["run_id"],
            "64-feature per-class artifact run ID": reduced_result.run_id,
            "Chosen feature set": (
                "reduced_64" if use_reduced_features else "all_71"
            ),
        }
    )

feature_set_comparison = pd.DataFrame(comparison_rows)
if len(feature_set_comparison) != 10:
    raise AssertionError("Expected ten paired feature-set comparisons.")
if feature_set_comparison[["Model family", "Weighting mode"]].duplicated().any():
    raise AssertionError("A paired configuration appears more than once.")
display(feature_set_comparison)

all_real_configuration_results = pd.concat(
    [validation_results, feature_ablation_results], ignore_index=True
).loc[lambda frame: frame["model_family"] != "DummyClassifier"]
selected_configuration_per_family = (
    all_real_configuration_results
    .sort_values("macro_f1", ascending=False)
    .drop_duplicates(subset="model_family", keep="first")
)
if len(selected_configuration_per_family) != 5:
    raise AssertionError("Expected one selected configuration for each real family.")
print("Best validation configuration from each real model family:")
display(
    selected_configuration_per_family[
        [
            "model_family",
            "weighting_mode",
            "feature_set",
            "macro_f1",
            "binary_attack_recall",
            "latency_p99_ms",
            "throughput_flows_per_second",
        ]
    ]
)

,Model family,Weighting mode,71-feature macro F1,64-feature macro F1,Macro F1 difference (64 - 71),71-feature p99 latency (ms),64-feature p99 latency (ms),p99 latency difference (64 - 71),71-feature throughput,64-feature throughput,Throughput difference (64 - 71),71 source features,71 transformed features,64 source features,64 transformed features,71-feature per-class artifact run ID,64-feature per-class artifact run ID,Chosen feature set
0,SGDClassifier,unweighted,0.391504,0.383278,-0.008226,2.323363,2.636099,0.312736,1.065982e+06,1.047430e+06,-18552.452631,71,73,64,66,bd0bd8f5c6704072a62bc62bd63501f7,5f4aff4cb50e46f6ae0ef16f6b51d708,all_71
1,SGDClassifier,balanced,0.452873,0.447660,-0.005213,2.399724,2.213198,-0.186526,1.072248e+06,1.083334e+06,11086.339199,71,73,64,66,92d4adc9a14a45b0b7151b35f487024d,cddc80dee65645eeb67f65d7f3c89aec,all_71
2,DecisionTreeClassifier,unweighted,0.809482,0.811481,0.001999,1.708117,1.739735,0.031618,1.467317e+06,1.490211e+06,22893.819770,71,73,64,66,4a8bd2b338524bc696eaa8d2091b3a3b,03c6738846414fe39dbf16a9bb2c5575,reduced_64
3,DecisionTreeClassifier,balanced,0.942137,0.942255,0.000118,1.851507,1.500515,-0.350992,1.288789e+06,1.608798e+06,320008.857931,71,73,64,66,556c47f26e9740abb748777aee46e419,bfd90d9524df44cf928d054a34a4b8a8,reduced_64
4,RandomForestClassifier,unweighted,0.856176,0.884431,0.028255,33.786842,34.225024,0.438182,1.996887e+05,1.999531e+05,264.405832,71,73,64,66,6113b02899214082abe8468144d2d56d,4f00556d9b664614911921f2751add6e,reduced_64
5,RandomForestClassifier,balanced,0.895785,0.948437,0.052653,33.956476,33.906162,-0.050314,2.000011e+05,1.887238e+05,-11277.293181,71,73,64,66,b5c29cbe221d4a20ad8f08cb29b51a25,e426da11cca14902a3e502cdd47c22f3,reduced_64
6,HistGradientBoostingClassifier,unweighted,0.950572,0.950144,-0.000428,28.950075,30.699401,1.749326,6.053683e+04,5.931190e+04,-1224.931895,71,73,64,66,5a8654a12b534ae4b1c0496816f4ed78,79abfa161e964811a7fbe3adf30849f5,all_71
7,HistGradientBoostingClassifier,balanced,0.961751,0.967029,0.005278,18.953830,18.948630,-0.005200,1.038161e+05,1.003926e+05,-3423.478639,71,73,64,66,672b352e49c04ad7bedead13035f695c,53cbe1eeec8d40c597db63190993b6bd,reduced_64
8,MLPClassifier,unweighted,0.696669,0.779481,0.082812,2.276155,2.256447,-0.019708,5.555241e+05,5.682270e+05,12702.942655,71,73,64,66,cfb05a1d134e481aa5e4e1ee059d38ac,b65293eea9d94ae6927dd685eae40c21,reduced_64
9,MLPClassifier,balanced,0.597323,0.625217,0.027894,2.362549,2.138233,-0.224316,5.708564e+05,5.857119e+05,14855.498640,71,73,64,66,f27d5257554647bebe366c179cacd308,f6df06f2c8e944a1a31d3f6a472404df,reduced_64


Best validation configuration from each real model family:


,model_family,weighting_mode,feature_set,macro_f1,binary_attack_recall,latency_p99_ms,throughput_flows_per_second
18,HistGradientBoostingClassifier,balanced,reduced_64,0.967029,0.999854,18.948630,1.003926e+05
16,RandomForestClassifier,balanced,reduced_64,0.948437,0.999607,33.906162,1.887238e+05
14,DecisionTreeClassifier,balanced,reduced_64,0.942255,0.999528,1.500515,1.608798e+06
19,MLPClassifier,unweighted,reduced_64,0.779481,0.995507,2.256447,5.682270e+05
8,SGDClassifier,balanced,all_71,0.452873,0.985980,2.399724,1.072248e+06


Feature and weighting selection remain model-specific. A reduced schema or balanced loss adopted by one family does not force another family to make the same choice. Dummy remains a single reference because repeating it cannot provide feature-relevance evidence. No test predictions or model serialization occur in this notebook.

## 9. Validation-only acceptance checks and next stages

This last cell summarizes the validation-only invariants. After these experiments, XGBoost must be tested with both feature sets, the strongest candidates must be tuned, and all decisions must be locked before the 80% training partition is refitted and the untouched test set is evaluated once.

In [12]:
required_validation_metrics = {
    "macro_f1",
    "accuracy_reference",
    "binary_attack_recall",
    "training_time_seconds",
    "latency_p50_ms",
    "latency_p95_ms",
    "latency_p99_ms",
    "throughput_flows_per_second",
}
if not required_validation_metrics.issubset(validation_results.columns):
    raise AssertionError("Validation results are missing required MLflow metrics.")
if not required_validation_metrics.issubset(feature_ablation_results.columns):
    raise AssertionError("Ablation results are missing required MLflow metrics.")
if set(split_distribution.index) != set(LABEL_ORDER):
    raise AssertionError("The partition report does not contain all fixed labels.")
if validation_results["evaluation_stage"].ne("validation").any():
    raise AssertionError("A validation result has the wrong stage tag.")
if feature_ablation_results["evaluation_stage"].ne("feature_ablation").any():
    raise AssertionError("An ablation result has the wrong stage tag.")
if len(MODEL_SPECS) != 11 or len(feature_ablation_results) != 10:
    raise AssertionError("Unexpected baseline or ablation run count.")
if feature_ablation_results["feature_set"].ne("reduced_64").any():
    raise AssertionError("Every ablation run must use reduced_64.")
if feature_ablation_results["source_feature_count"].ne(64).any():
    raise AssertionError("An ablation run has the wrong source feature count.")
if feature_ablation_results["transformed_feature_count"].ne(66).any():
    raise AssertionError("An ablation run has the wrong transformed feature count.")
combined_experiment_results = pd.concat(
    [validation_results, feature_ablation_results], ignore_index=True
)
if len(combined_experiment_results) != 21:
    raise AssertionError("Expected 21 total baseline and ablation runs.")
if combined_experiment_results[
    ["model_key", "weighting_mode", "feature_set"]
].duplicated().any():
    raise AssertionError("The full experiment matrix contains a duplicate configuration.")
if combined_experiment_results["fit_split_fingerprint"].nunique() != 1:
    raise AssertionError("Experiment runs used different fitting rows.")
if combined_experiment_results["evaluation_split_fingerprint"].nunique() != 1:
    raise AssertionError("Experiment runs used different validation rows.")
if combined_experiment_results["timing_input_fingerprint"].nunique() != 1:
    raise AssertionError("Experiment runs used different timing inputs.")

acceptance_summary = pd.Series(
    {
        "All 15 labels in every partition": True,
        "Partition indices are disjoint": True,
        "Macro F1 independently verified": True,
        "Confusion-matrix label order fixed": True,
        "Normalized non-empty rows sum to one": True,
        "All real validation runs beat Dummy": True,
        "Weighted and unweighted runs share partitions": True,
        "Eleven 71-feature baselines completed": True,
        "Ten paired 64-feature real-model runs completed": True,
        "Every pipeline one-hot encoded Protocol": True,
        "Source and transformed feature counts verified": True,
        "No test predictions performed": True,
        "Required operational metrics logged": True,
    },
    name="Passed",
)
display(acceptance_summary.to_frame())
print("The validation-only baseline and feature-ablation workflow is complete.")
print("Next: XGBoost with both feature sets, controlled tuning, then one final test.")

,Passed
All 15 labels in every partition,True
Partition indices are disjoint,True
Macro F1 independently verified,True
Confusion-matrix label order fixed,True
Normalized non-empty rows sum to one,True
All real validation runs beat Dummy,True
Weighted and unweighted runs share partitions,True
Eleven 71-feature baselines completed,True
Ten paired 64-feature real-model runs completed,True
Every pipeline one-hot encoded Protocol,True


The validation-only baseline and feature-ablation workflow is complete.
Next: XGBoost with both feature sets, controlled tuning, then one final test.
